In [ ]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from astroquery.gaia import Gaia
import pandas as pd
import plotly.io as pio

Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

In [ ]:
!pip install nbformat --upgrade

In [ ]:
t = Table.read('GalahC.fits')
df = t.to_pandas()
#Picking stars that fulfill parameters to be a giant
giants = df[(df['teff'] < 5500) & (df['logg'] < 3.0) & (df['flag_sp'] == 0)]
print(f"Total giants found: {len(giants)}")                                             #Print number of giants that fulfill condition
giants_sample = giants.sample(40, random_state=42)                                      #Choose 40 random giants from this selection
print(giants_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])       
giants_sample.to_csv('giants.csv', index=False)                                         

subgiants = df[(df['teff'].between(5000, 6500)) & (df['logg'].between(3.0, 4.0))]
print(f"Total subgiants found: {len(subgiants)}")
subgiants_sample = subgiants.sample(40, random_state=42)
print(subgiants_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
subgiants_sample.to_csv('subgiants.csv', index=False)

soba = df[(df['teff'] > 7500) & (df['logg'] > 3.5)]
print(f"Total soba found: {len(soba)}")
soba_sample = soba.sample(40, random_state=42)
print(soba_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
soba_sample.to_csv('soba.csv', index=False)

sfg = df[(`df['teff'].between (5200, 7500)) & (df['logg'] > 4.0)]
print(f"Total sfg found: {len(sfg)}")
sfg_sample = sfg.sample(40, random_state=42)
print(sfg_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
sfg_sample.to_csv('sfg.csv', index=False)

skm = df[(df['teff'] < 5200) & (df['logg'] > 4.0)]
print(f"Total skm found: {len(skm)}")
skm_sample = skm.sample(40, random_state=42)
print(skm_sample[['sobject_id', 'teff', 'logg', 'fe_h', 'gaiadr3_source_id']])
skm_sample.to_csv('skm.csv', index=False)

In [ ]:
# Query selected giants from the Gaia Catalogue, extract information and turn to csv
# Load your saved giants
giants = pd.read_csv('giants.csv')

# Get Gaia source IDs
source_ids = giants['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('giantsgaiadistance.csv', index=False)
print("Saved!")

In [ ]:
# Convert giants given ra, dec and distance into cartesian
df = pd.read_csv('giantsgaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total giants:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('giantscartesian.csv', index=False)

In [ ]:
# Query selected subgiants from the Gaia Catalogue, extract information and turn to csv
# Load your saved subgiants
subgiants = pd.read_csv('subgiants.csv')

# Get Gaia source IDs
source_ids = subgiants['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('subgiantsgaiadistance.csv', index=False)
print("Saved!")

In [ ]:
# Convert subgiants given ra, dec and distance into cartesian
df = pd.read_csv('subgiantsgaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total subgiants:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('subgiantscartesian.csv', index=False)

In [ ]:
# Query selected soba from the Gaia Catalogue, extract information and turn to csv
# Load your saved soba
soba = pd.read_csv('soba.csv')

# Get Gaia source IDs
source_ids = soba['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('sobagaiadistance.csv', index=False)
print("Saved!")

In [ ]:
# Convert soba given ra, dec and distance into cartesian
df = pd.read_csv('sobagaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total soba:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('sobacartesian.csv', index=False)

In [ ]:
# Query selected skm from the Gaia Catalogue, extract information and turn to csv
# Load your saved skm
skm = pd.read_csv('skm.csv')

# Get Gaia source IDs
source_ids = skm['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('skmgaiadistance.csv', index=False)
print("Saved!")

In [ ]:
# Convert skm given ra, dec and distance into cartesian
df = pd.read_csv('skmgaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total skm:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('skmcartesian.csv', index=False)

In [ ]:
# Query selected sfg from the Gaia Catalogue, extract information and turn to csv
# Load your saved sfg
sfg = pd.read_csv('sfg.csv')

# Get Gaia source IDs
source_ids = sfg['gaiadr3_source_id'].dropna().astype(int).tolist()

ids_str = ','.join(map(str, source_ids))

# Query Gaia DR3
query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, pm, distance_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

job = Gaia.launch_job(query)
result = job.get_results()
print(result) 

result_df = result.to_pandas()
result_df.to_csv('sfggaiadistance.csv', index=False)
print("Saved!")

In [ ]:
# Convert sfg given ra, dec and distance into cartesian
df = pd.read_csv('sfggaiadistance.csv')

# Filter to only rows that have all three values
has_data = df[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
clean = df[has_data].copy()

print(f"Total skm:           {len(df)}")
print(f"With complete data:     {len(clean)}")
print(f"Missing distance:       {len(df) - len(clean)}")

# Convert all at once
coords = SkyCoord(
    ra=clean['ra'].values * u.deg,
    dec=clean['dec'].values * u.deg,
    distance=clean['distance_gspphot'].values * u.pc,
    frame='icrs'
)

# Extract cartesian
clean['x_pc'] = coords.cartesian.x.to(u.pc).value
clean['y_pc'] = coords.cartesian.y.to(u.pc).value
clean['z_pc'] = coords.cartesian.z.to(u.pc).value

print(clean[['source_id', 'ra', 'dec', 'distance_gspphot', 'x_pc', 'y_pc', 'z_pc']])
clean.to_csv('sfgcartesian.csv', index=False)

In [ ]:
# Load all your star type CSVs
giants      = pd.read_csv('giantscartesian.csv')
subgiants      = pd.read_csv('subgiantscartesian.csv')
mainsoba = pd.read_csv('sobacartesian.csv')
mainskm = pd.read_csv('skmcartesian.csv')
mainsfg = pd.read_csv('sfgcartesian.csv')

# Label each group
giants['region']      = 'Giants'
subgiants['region'] = 'Subgaints'
mainsoba['region'] = 'Main Sequence O/B/A'
mainskm['region'] = 'Main Sequence K/M'
mainsfg['region'] = 'Main Sequence F/G'

# Combine into one dataframe
all_stars = pd.concat([giants, subgiants, mainsoba, mainskm, mainsfg], ignore_index=True)

# Define colours per type
colours = {
    'Giants':   'red',
    'Subgiants': 'orange',
    'Main Sequence O/B/A': 'white',
    'Main Sequence F/G': 'yellow',
    'Main Sequence K/M': 'brown',
}

# Plot
fig = go.Figure()

for region, group in all_stars.groupby('region'):
    # Drop rows missing cartesian coords
    group = group.dropna(subset=['x_pc', 'y_pc', 'z_pc'])
    
    fig.add_trace(go.Scatter3d(
        x=group['x_pc'],
        y=group['y_pc'],
        z=group['z_pc'],
        mode='markers',
        name=region,
        marker=dict(
            size=4,
            color=colours.get(region, 'grey'),
            opacity=0.8
        ),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'x: %{x:.1f} pc<br>'
            'y: %{y:.1f} pc<br>'
            'z: %{z:.1f} pc'
        ),
        text=group['source_id']
    ))

# Add Sun at origin (since we used ICRS/Earth centred coords)
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=8, color='yellow', symbol='diamond'),
    name='Earth'
))

fig.update_layout(
    title='3D Star Positions',
    scene=dict(
        bgcolor='black',
        xaxis=dict(backgroundcolor='black', gridcolor='grey', title='X (pc)'),
        yaxis=dict(backgroundcolor='black', gridcolor='grey', title='Y (pc)'),
        zaxis=dict(backgroundcolor='black', gridcolor='grey', title='Z (pc)'),
    ),
    paper_bgcolor='black',
    font_color='white'
)

fig.show()